In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/augmentation/images/images.zip -d /content/images.zip
!unzip -q images.zip

!rm /content/images.zip

import os
A=os.listdir('/content')
A

import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
s='/content/images/*'
path=glob.glob(s)
import re
def sorted_alphanumeric(data):
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [ convert(c) for c in re.split('([0-9]+)', key) ]
    return sorted(data, key=alphanum_key)

path1=sorted_alphanumeric(path)
path1 = [item.replace('p','_') for item in path1]
data=pd.read_excel('/content/drive/MyDrive/augmentation/data_c.xlsx')
data
len(path1)
print(path1)
type(path1)
path1=np.array(path1)
path1.shape
path1[0]

Mounted at /content/drive


In [ ]:
path1=np.array(path1)
data=np.array(data)

In [ ]:
df=pd.DataFrame({'imgpath':path1,'Porosity':data[:,0],'throat radius':data[:,1],'pore radius':data[:,2],'pore_connection_number':data[:,3],'pore shape factor':data[:,4]})

In [ ]:
df_new=df[df['Porosity'] <0.35 ]
df_new.shape
df_new

In [ ]:
df_new_two=df_new.iloc[::2,:]
df_new_two

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
data1=np.array(df_new_two)
scaler = StandardScaler()
data1= scaler.fit_transform(data1[:,1:])
print(data1)
print(type(data1))
data1.shape

In [ ]:
df=pd.DataFrame({'imgpath':df_new_two['imgpath'],'Porosity':data1[:,0],'throat radius':data1[:,1],'pore radius':data1[:,2],'pore_connection_number':data1[:,3],'pore shape factor':data1[:,4]})
df

In [ ]:
data_t_general=df.sample(frac=0.9,random_state=1)
df_valid=data_t_general.sample(frac=0.25,random_state=2)
df_train=data_t_general.loc[~data_t_general.index.isin(df_valid.index)]
df_test =df.loc[~df.index.isin(data_t_general.index)]
len(df_train)

In [ ]:
####no_augment####

In [ ]:
def Data_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img = []
                y_batch=[]
                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    Porosity=np.array(df.Porosity)[idd]
                    throat_radius=np.array(df['throat radius'])[idd]
                    pore_radius=np.array(df['pore radius'])[idd]
                    pore_connection_number=np.array(df.pore_connection_number)[idd]
                    pore_shape_factor=np.array(df['pore shape factor'])[idd]
                    y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
                    y=np.array([y_1])
                    x_batch_img.append(img_1)
                    y_batch.append(y)


                x_batch_img = np.array(x_batch_img)
                y_batch= np.array(y_batch)
                y_batch=y_batch.reshape(-1,5)
              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img , y_batch

In [ ]:
#from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
import cv2
def apply_augmentation(image):
    final_augmented = []
    final_augmented.append(image)
    for i in range(3):
        rotation_angle = (i+1) * 90
        rotation_matrix = cv2.getRotationMatrix2D((100 // 2, 100 // 2), rotation_angle, 1.0)
        rotated_slices = [cv2.warpAffine(slice_2d, rotation_matrix, (100, 100)) for slice_2d in image]
        augmented_image = np.stack(rotated_slices, axis=0)
        final_augmented.append(augmented_image)
    for j in range(2):
        flipped_slices = [cv2.flip(slice_2d, j) for slice_2d in image]
        augmented_image_2 = np.stack(flipped_slices, axis=0)
        final_augmented.append(augmented_image_2)
    for k in range(2):
        rotation_matrix90 = cv2.getRotationMatrix2D((100 // 2, 100 // 2), 90, 1.0)
        rotated_slices90 = [cv2.warpAffine(slice_2d, rotation_matrix90, (100, 100)) for slice_2d in image]
        augmented_image90 = np.stack(rotated_slices90, axis=0)
        flipped_rotate_slices = [cv2.flip(slice_2d, k) for slice_2d in augmented_image90]
        augmented_image90_f =  np.stack(flipped_rotate_slices, axis=0)
        final_augmented.append(augmented_image90_f)

    return final_augmented


In [ ]:
#######augment######

In [ ]:
#from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
class Data_generator(tf.keras.utils.Sequence):
  def __init__(self, data, batch_size=4, dim=(100,100,100), channels=1, shuffle=False, augment=False):
    self.data = data
    self.batch_size=batch_size
    self.dim=dim
    self.channels=channels
    self.shuffle=shuffle
    self.on_epoch_end()

  def __len__(self):
    return int(np.floor(len(self.data) / self.batch_size))

  def __getitem__(self, index):
    indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
    # Generate data
    X, y = self.__data_generation(indexes)
    return X, y

  def on_epoch_end(self):

    'Updates indexes after each epoch'
    self.indexes = np.arange(len(self.data))
    if self.shuffle == True:
        np.random.shuffle(self.indexes)

  def __data_generation(self, indexes):
      x_batch_img = []
      y_batch = []

      for idd in indexes:
          img = open(np.array(self.data.imgpath)[idd],'rb').read()
          img_1 = np.frombuffer(img,dtype=np.uint8)
          #img_1 = img_1.reshape(100,100,100,1)
          img_1 = img_1.reshape(100,100,100)
          img_1 = apply_augmentation(img_1)
          img_1 = [each_image.reshape(100,100,100,1) for each_image in img_1]
          x_batch_img.extend(img_1)
          Porosity=np.array(self.data.Porosity)[idd]
          throat_radius=np.array(self.data['throat radius'])[idd]
          pore_radius=np.array(self.data['pore radius'])[idd]
          pore_connection_number=np.array(self.data.pore_connection_number)[idd]
          pore_shape_factor=np.array(self.data['pore shape factor'])[idd]
          y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
          y=np.array([y_1])
          for i in range(8):
            y_batch.append(y)


      x_batch_img = np.array(x_batch_img)
      y_batch = np.array(y_batch)
      y_batch = y_batch.reshape(-1,5)

      return x_batch_img, y_batch

In [ ]:
from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
def Data_generator(df, batch_size, augment=False):
  for start in range(0, df.shape[0], batch_size):
      x_batch_img = []
      y_batch = []
      end = min(start + batch_size, df.shape[0])

      for idd in range(start,end):
          img = open(np.array(df.imgpath)[idd],'rb').read()
          img_1 = np.frombuffer(img,dtype=np.uint8)
          #img_1 = img_1.reshape(100,100,100,1)
          img_1 = img_1.reshape(100,100,100)
          img_1 = apply_augmentation(img_1)
          img_1 = [each_image.reshape(100,100,100,1) for each_image in img_1]
          x_batch_img.extend(img_1)
          Porosity=np.array(df.Porosity)[idd]
          throat_radius=np.array(df['throat radius'])[idd]
          pore_radius=np.array(df['pore radius'])[idd]
          pore_connection_number=np.array(df.pore_connection_number)[idd]
          pore_shape_factor=np.array(df['pore shape factor'])[idd]
          y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
          y=np.array([y_1])
          for i in range(8):
            y_batch.append(y)


      x_batch_img = np.array(x_batch_img)
      y_batch = np.array(y_batch)
      y_batch = y_batch.reshape(-1,5)




      yield x_batch_img, y_batch

In [ ]:
batch_size=10
train_gen= Data_generator(df_train,batch_size)
valid_gen= Data_generator(df_valid,batch_size)
# train_gen_pre=Data_predict_generator(df_train,batch_size)
# valid_gen_pre=Data_predict_generator(df_valid,batch_size)
test_gen= Data_generator(df_test,batch_size)
# test_gen_pre=Data_predict_generator(df_test,batch_size)

In [ ]:
import math
ntrain, nvalid, ntest = df_train.shape[0], df_valid.shape[0],  df_test.shape[0]
nbatches_train=math.ceil(df_train.shape[0]/batch_size)
nbatches_valid=math.ceil(df_valid.shape[0]/batch_size)
nbatches_test=math.ceil( df_test.shape[0]/batch_size)
nworkers=1

In [ ]:
print(nbatches_valid,nbatches_train,nbatches_test)

90 268 40


In [ ]:
batch_size=4
train_gen= Data_generator(df_train,batch_size)
valid_gen= Data_generator(df_valid,batch_size)
# train_gen_pre=Data_predict_generator(df_train,batch_size)
# valid_gen_pre=Data_predict_generator(df_valid,batch_size)
test_gen= Data_generator(df_test,batch_size)
# test_gen_pre=Data_predict_generator(df_test,batch_size)

In [ ]:
import math
ntrain, nvalid, ntest = df_train.shape[0], df_valid.shape[0],  df_test.shape[0]
nbatches_train=math.ceil(df_train.shape[0]/batch_size)
nbatches_valid=math.ceil(df_valid.shape[0]/batch_size)
nbatches_test=math.ceil( df_test.shape[0]/batch_size)
nworkers=1

In [ ]:
from keras.models import Sequential
from keras.layers import Conv3D,MaxPooling3D,Dropout,Flatten,Dense,BatchNormalization,ReLU
from tensorflow.keras import initializers
from keras.callbacks import *
import tensorflow as tf

In [ ]:
model=Sequential()
model.add(Conv3D(16,kernel_size=(7,7,7),input_shape=(100,100,100,1),padding="same",activation='relu',kernel_initializer=initializers.RandomNormal(stddev=0.01),))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(32,kernel_size=(5,5,5),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(64,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(128,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(256,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
#model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(1024,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(512,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(5))
model.summary()

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/sensivity_augment/50%fulldata set/no_augment/training_model_weights", exist_ok=True)
ckpt_callback = ModelCheckpoint(filepath='/content/drive/MyDrive/sensivity_augment/50%fulldata set/no_augment/training_model_weights/weights.{epoch:03d}.keras', monitor='val_loss')

In [ ]:
csvlogger=CSVLogger('/content/drive/MyDrive/sensivity_augment/50%fulldata set/no_augment/Training_70.log')

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(opt,loss='mse',metrics=['mse'])

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger])

Epoch 1/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 188s 567ms/step - loss: 3.7380 - mse: 3.7380 - val_loss: 3.9185 - val_mse: 3.9185
Epoch 2/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 130s 486ms/step - loss: 2.0766 - mse: 2.0766 - val_loss: 0.8457 - val_mse: 0.8457
Epoch 3/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 130s 484ms/step - loss: 1.4807 - mse: 1.4807 - val_loss: 1.0679 - val_mse: 1.0679
Epoch 4/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 130s 484ms/step - loss: 1.1357 - mse: 1.1357 - val_loss: 15.4388 - val_mse: 15.4388
Epoch 5/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 130s 485ms/step - loss: 0.8385 - mse: 0.8385 - val_loss: 2.0468 - val_mse: 2.0468
Epoch 6/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 130s 485ms/step - loss: 0.6572 - mse: 0.6572 - val_loss: 0.8469 - val_mse: 0.8469
Epoch 7/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 130s 484ms/step - loss: 0.5310 - mse: 0.5310 - val_loss: 1.5731 - val_mse: 1.5731
Epoch 8/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 130s 485ms/step - loss: 0.4650 - mse: 0.4650 - val_loss: 0.4766 - val_mse: 0.4766
Epoch 9/200
26

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/no_augment/training_model_weights/weights.088.keras')

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger],initial_epoch=88)

Epoch 89/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 177s 544ms/step - loss: 0.1458 - mse: 0.1458 - val_loss: 0.1901 - val_mse: 0.1901
Epoch 90/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 128s 479ms/step - loss: 0.1462 - mse: 0.1462 - val_loss: 0.1805 - val_mse: 0.1805
Epoch 91/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 131s 488ms/step - loss: 0.1462 - mse: 0.1462 - val_loss: 0.1781 - val_mse: 0.1781
Epoch 92/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 131s 489ms/step - loss: 0.1458 - mse: 0.1458 - val_loss: 0.2164 - val_mse: 0.2164
Epoch 93/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 131s 488ms/step - loss: 0.1455 - mse: 0.1455 - val_loss: 0.1754 - val_mse: 0.1754
Epoch 94/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 131s 489ms/step - loss: 0.1425 - mse: 0.1425 - val_loss: 0.1771 - val_mse: 0.1771
Epoch 95/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 131s 488ms/step - loss: 0.1452 - mse: 0.1452 - val_loss: 0.1781 - val_mse: 0.1781
Epoch 96/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 132s 491ms/step - loss: 0.1457 - mse: 0.1457 - val_loss: 0.2000 - val_mse: 0.2000
Epoch 97

In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/no_augment/training_model_weights/weights.131.keras')

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger],initial_epoch=131)

Epoch 132/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 189s 580ms/step - loss: 0.1334 - mse: 0.1334 - val_loss: 0.1634 - val_mse: 0.1634
Epoch 133/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 133s 497ms/step - loss: 0.1376 - mse: 0.1376 - val_loss: 0.1625 - val_mse: 0.1625
Epoch 134/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 132s 494ms/step - loss: 0.1373 - mse: 0.1373 - val_loss: 0.1877 - val_mse: 0.1877
Epoch 135/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 133s 495ms/step - loss: 0.1361 - mse: 0.1361 - val_loss: 0.1655 - val_mse: 0.1655
Epoch 136/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 133s 497ms/step - loss: 0.1334 - mse: 0.1334 - val_loss: 0.1789 - val_mse: 0.1789
Epoch 137/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 132s 494ms/step - loss: 0.1350 - mse: 0.1350 - val_loss: 0.1906 - val_mse: 0.1906
Epoch 138/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 133s 495ms/step - loss: 0.1332 - mse: 0.1332 - val_loss: 0.1631 - val_mse: 0.1631
Epoch 139/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 132s 494ms/step - loss: 0.1336 - mse: 0.1336 - val_loss: 0.1699 - val_mse: 0.1699


In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/no_augment/training_model_weights/weights.131.keras')

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger],initial_epoch=131)

Epoch 132/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 197s 601ms/step - loss: 0.1352 - mse: 0.1352 - val_loss: 0.1790 - val_mse: 0.1790
Epoch 133/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 136s 508ms/step - loss: 0.1343 - mse: 0.1343 - val_loss: 0.1818 - val_mse: 0.1818
Epoch 134/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 136s 507ms/step - loss: 0.1371 - mse: 0.1371 - val_loss: 0.1897 - val_mse: 0.1897
Epoch 135/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 136s 509ms/step - loss: 0.1341 - mse: 0.1341 - val_loss: 0.1797 - val_mse: 0.1797
Epoch 136/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 136s 506ms/step - loss: 0.1335 - mse: 0.1335 - val_loss: 0.1673 - val_mse: 0.1673
Epoch 137/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 136s 507ms/step - loss: 0.1337 - mse: 0.1337 - val_loss: 0.1634 - val_mse: 0.1634
Epoch 138/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 136s 506ms/step - loss: 0.1362 - mse: 0.1362 - val_loss: 0.1957 - val_mse: 0.1957
Epoch 139/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 136s 506ms/step - loss: 0.1351 - mse: 0.1351 - val_loss: 0.1954 - val_mse: 0.1954


In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/no_augment/training_model_weights/weights.182.keras')

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger],initial_epoch=182)

Epoch 183/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 192s 572ms/step - loss: 0.1302 - mse: 0.1302 - val_loss: 0.1635 - val_mse: 0.1635
Epoch 184/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 131s 488ms/step - loss: 0.1283 - mse: 0.1283 - val_loss: 0.1586 - val_mse: 0.1586
Epoch 185/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 131s 489ms/step - loss: 0.1273 - mse: 0.1273 - val_loss: 0.1772 - val_mse: 0.1772
Epoch 186/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 131s 487ms/step - loss: 0.1291 - mse: 0.1291 - val_loss: 0.1607 - val_mse: 0.1607
Epoch 187/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 130s 487ms/step - loss: 0.1295 - mse: 0.1295 - val_loss: 0.1651 - val_mse: 0.1651
Epoch 188/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 131s 488ms/step - loss: 0.1264 - mse: 0.1264 - val_loss: 0.1585 - val_mse: 0.1585
Epoch 189/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 131s 488ms/step - loss: 0.1267 - mse: 0.1267 - val_loss: 0.1588 - val_mse: 0.1588
Epoch 190/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 130s 487ms/step - loss: 0.1269 - mse: 0.1269 - val_loss: 0.1966 - val_mse: 0.1966


In [ ]:
###############Augmenttttt#######################

In [ ]:
#from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
import cv2
def apply_augmentation(image):
    final_augmented = []
    final_augmented.append(image)
    for i in range(3):
        rotation_angle = (i+1) * 90
        rotation_matrix = cv2.getRotationMatrix2D((100 // 2, 100 // 2), rotation_angle, 1.0)
        rotated_slices = [cv2.warpAffine(slice_2d, rotation_matrix, (100, 100)) for slice_2d in image]
        augmented_image = np.stack(rotated_slices, axis=0)
        final_augmented.append(augmented_image)
    for j in range(2):
        flipped_slices = [cv2.flip(slice_2d, j) for slice_2d in image]
        augmented_image_2 = np.stack(flipped_slices, axis=0)
        final_augmented.append(augmented_image_2)
    for k in range(2):
        rotation_matrix90 = cv2.getRotationMatrix2D((100 // 2, 100 // 2), 90, 1.0)
        rotated_slices90 = [cv2.warpAffine(slice_2d, rotation_matrix90, (100, 100)) for slice_2d in image]
        augmented_image90 = np.stack(rotated_slices90, axis=0)
        flipped_rotate_slices = [cv2.flip(slice_2d, k) for slice_2d in augmented_image90]
        augmented_image90_f =  np.stack(flipped_rotate_slices, axis=0)
        final_augmented.append(augmented_image90_f)

    return final_augmented


In [ ]:
#from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
class Data_generator(tf.keras.utils.Sequence):
  def __init__(self, data, batch_size=4, dim=(100,100,100), channels=1, shuffle=False, augment=False):
    self.data = data
    self.batch_size=batch_size
    self.dim=dim
    self.channels=channels
    self.shuffle=shuffle
    self.on_epoch_end()

  def __len__(self):
    return int(np.floor(len(self.data) / self.batch_size))

  def __getitem__(self, index):
    indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
    # Generate data
    X, y = self.__data_generation(indexes)
    return X, y

  def on_epoch_end(self):

    'Updates indexes after each epoch'
    self.indexes = np.arange(len(self.data))
    if self.shuffle == True:
        np.random.shuffle(self.indexes)

  def __data_generation(self, indexes):
      x_batch_img = []
      y_batch = []

      for idd in indexes:
          img = open(np.array(self.data.imgpath)[idd],'rb').read()
          img_1 = np.frombuffer(img,dtype=np.uint8)
          #img_1 = img_1.reshape(100,100,100,1)
          img_1 = img_1.reshape(100,100,100)
          img_1 = apply_augmentation(img_1)
          img_1 = [each_image.reshape(100,100,100,1) for each_image in img_1]
          x_batch_img.extend(img_1)
          Porosity=np.array(self.data.Porosity)[idd]
          throat_radius=np.array(self.data['throat radius'])[idd]
          pore_radius=np.array(self.data['pore radius'])[idd]
          pore_connection_number=np.array(self.data.pore_connection_number)[idd]
          pore_shape_factor=np.array(self.data['pore shape factor'])[idd]
          y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
          y=np.array([y_1])
          for i in range(8):
            y_batch.append(y)


      x_batch_img = np.array(x_batch_img)
      y_batch = np.array(y_batch)
      y_batch = y_batch.reshape(-1,5)

      return x_batch_img, y_batch

In [ ]:
batch_size=4
train_gen= Data_generator(df_train,batch_size)
valid_gen= Data_generator(df_valid,batch_size)
train_gen_pre=Data_predict_generator(df_train,batch_size)
valid_gen_pre=Data_predict_generator(df_valid,batch_size)
test_gen= Data_generator(df_test,batch_size)
test_gen_pre=Data_predict_generator(df_test,batch_size)

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/training_model_weights", exist_ok=True)
ckpt_callback = ModelCheckpoint(filepath='/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/training_model_weights/weights.{epoch:03d}.keras')
#cp_callback=ModelCheckpoint(filepath=checkpoint_path,save_weights_only=True,verbose=1)

In [ ]:
csvlogger=CSVLogger('/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/Training_175.log')

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(opt,loss='mse',metrics=['mse'])

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/shared/training_model_weights/weights.060.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=60)

Epoch 61/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1086s 2s/step - loss: 0.2835 - mse: 0.2835 - val_loss: 0.2019 - val_mse: 0.2019
Epoch 62/200
260/668 ━━━━━━━━━━━━━━━━━━━━ 9:10 1s/step - loss: 0.2670 - mse: 0.2670

In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/shared/training_model_weights/weights.069.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=69)

Epoch 70/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1068s 1s/step - loss: 0.2869 - mse: 0.2869 - val_loss: 0.1948 - val_mse: 0.1948
Epoch 71/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 979s 1s/step - loss: 0.2950 - mse: 0.2950 - val_loss: 0.1933 - val_mse: 0.1933
Epoch 72/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1033s 2s/step - loss: 0.2758 - mse: 0.2758 - val_loss: 0.2185 - val_mse: 0.2185
Epoch 73/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1034s 2s/step - loss: 0.2962 - mse: 0.2962 - val_loss: 0.1923 - val_mse: 0.1923
Epoch 74/200
514/668 ━━━━━━━━━━━━━━━━━━━━ 3:24 1s/step - loss: 0.3168 - mse: 0.3168

In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/shared/training_model_weights/weights.079.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=79)

Epoch 80/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1123s 2s/step - loss: 0.2844 - mse: 0.2844 - val_loss: 0.1910 - val_mse: 0.1910
Epoch 81/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1003s 1s/step - loss: 0.2914 - mse: 0.2914 - val_loss: 0.1955 - val_mse: 0.1955
Epoch 82/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1092s 2s/step - loss: 0.2842 - mse: 0.2842 - val_loss: 0.2611 - val_mse: 0.2611
Epoch 83/200
190/668 ━━━━━━━━━━━━━━━━━━━━ 10:51 1s/step - loss: 0.3143 - mse: 0.3143

In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/training_model_weights/weights.082.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=82)

Epoch 83/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1099s 2s/step - loss: 0.2775 - mse: 0.2775 - val_loss: 0.1917 - val_mse: 0.1917
Epoch 84/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1006s 2s/step - loss: 0.2914 - mse: 0.2914 - val_loss: 0.2148 - val_mse: 0.2148
Epoch 85/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1041s 2s/step - loss: 0.2893 - mse: 0.2893 - val_loss: 0.1885 - val_mse: 0.1885
Epoch 86/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1057s 2s/step - loss: 0.2812 - mse: 0.2812 - val_loss: 0.1922 - val_mse: 0.1922
Epoch 87/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1059s 2s/step - loss: 0.2697 - mse: 0.2697 - val_loss: 0.1952 - val_mse: 0.1952
Epoch 88/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1045s 1s/step - loss: 0.2719 - mse: 0.2719 - val_loss: 0.1913 - val_mse: 0.1913
Epoch 89/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1003s 1s/step - loss: 0.2772 - mse: 0.2772 - val_loss: 0.2068 - val_mse: 0.2068
Epoch 90/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1042s 1s/step - loss: 0.2900 - mse: 0.2900 - val_loss: 0.2142 - val_mse: 0.2142
Epoch 91/200
569/668 ━━━━━━━━━━━━━━━━

In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/training_model_weights/weights.095.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=95)

Epoch 96/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1067s 1s/step - loss: 0.2777 - mse: 0.2777 - val_loss: 0.2024 - val_mse: 0.2024
Epoch 97/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1065s 2s/step - loss: 0.2792 - mse: 0.2792 - val_loss: 0.1919 - val_mse: 0.1919
Epoch 98/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1027s 2s/step - loss: 0.2857 - mse: 0.2857 - val_loss: 0.2533 - val_mse: 0.2533
Epoch 99/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 971s 1s/step - loss: 0.2864 - mse: 0.2864 - val_loss: 0.1888 - val_mse: 0.1888
Epoch 100/200
631/668 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - loss: 0.2857 - mse: 0.2857

In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/shared/training_model_weights/weights.125.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=125)

Epoch 126/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1064s 1s/step - loss: 0.2800 - mse: 0.2800 - val_loss: 0.2093 - val_mse: 0.2093
Epoch 127/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1014s 1s/step - loss: 0.2831 - mse: 0.2831 - val_loss: 0.1954 - val_mse: 0.1954
Epoch 128/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 984s 1s/step - loss: 0.2743 - mse: 0.2743 - val_loss: 0.2138 - val_mse: 0.2138
Epoch 129/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 973s 1s/step - loss: 0.2771 - mse: 0.2771 - val_loss: 0.1898 - val_mse: 0.1898
Epoch 130/200
195/668 ━━━━━━━━━━━━━━━━━━━━ 10:25 1s/step - loss: 0.2509 - mse: 0.2509

In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/training_model_weights/weights.129.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=129)

Epoch 130/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1142s 2s/step - loss: 0.2633 - mse: 0.2633 - val_loss: 0.2008 - val_mse: 0.2008
Epoch 131/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1039s 2s/step - loss: 0.2634 - mse: 0.2634 - val_loss: 0.1930 - val_mse: 0.1930
Epoch 132/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1068s 2s/step - loss: 0.2694 - mse: 0.2694 - val_loss: 0.1997 - val_mse: 0.1997
Epoch 133/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1049s 2s/step - loss: 0.2876 - mse: 0.2876 - val_loss: 0.1865 - val_mse: 0.1865
Epoch 134/200
349/668 ━━━━━━━━━━━━━━━━━━━━ 7:21 1s/step - loss: 0.2910 - mse: 0.2910

In [ ]:
model= load_model('/content/drive/MyDrive/sensivity_augment/50%fulldata set/with_augment/shared/training_model_weights/weights.174.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=174)

Epoch 175/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1068s 1s/step - loss: 0.2842 - mse: 0.2842 - val_loss: 0.1892 - val_mse: 0.1892
Epoch 176/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 983s 1s/step - loss: 0.2832 - mse: 0.2832 - val_loss: 0.1883 - val_mse: 0.1883
Epoch 177/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1042s 1s/step - loss: 0.2712 - mse: 0.2712 - val_loss: 0.1876 - val_mse: 0.1876
Epoch 178/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1042s 1s/step - loss: 0.2830 - mse: 0.2830 - val_loss: 0.1850 - val_mse: 0.1850
Epoch 179/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 983s 1s/step - loss: 0.2737 - mse: 0.2737 - val_loss: 0.1895 - val_mse: 0.1895
Epoch 180/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1040s 1s/step - loss: 0.2711 - mse: 0.2711 - val_loss: 0.1844 - val_mse: 0.1844
Epoch 181/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 987s 1s/step - loss: 0.2796 - mse: 0.2796 - val_loss: 0.2018 - val_mse: 0.2018
Epoch 182/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1042s 1s/step - loss: 0.2573 - mse: 0.2573 - val_loss: 0.1905 - val_mse: 0.1905
Epoch 183/200
668/668 ━━━━━━━━━━━